# 1001 22일차

## 1. 임베딩

자연어를 모델이 연산할 수 있는 실수 벡터로 바꾸는 것

AI는 자연어를 그대로 연산할 수 없어서 숫자로 바꿔 넘겨야 함

| 단계 | 예 |
|---|---|
| 문장 | 나는 밥을 |
| 토큰화 | `[나는, 밥을]` |
| 정수 인코딩 | `[3, 17]` |
| 임베딩 | `[[0.21, -0.53, ...], [0.88, 0.10, ...]]` |

1. 토큰화 : 문장을 단어나 글자 단위로 자름
2. 정수 인코딩 : 토큰마다 번호를 붙임
3. 임베딩 : 번호 하나를 실수 여러 개짜리 벡터로 바꿈

- 수치화는 2번에서 이미 됨. 임베딩은 그다음 단계
- 번호는 이름표라 3번과 4번이 비슷한 뜻이라는 정보가 없음
- 임베딩 벡터의 값은 가중치라 학습하면서 뜻이 비슷한 단어끼리 가까워짐

### 1-1. 원핫의 한계

원핫으로 바꾸면 단어 하나가 단어 종류 수만큼의 칸이 되고, 그중 1칸만 1이고 나머지는 0임

| | 원핫 (`to_categorical`) | 임베딩 (`Embedding(31, 10)`) |
|---|---|---|
| 모양 | `(15, 5, 31)` | `(15, 5, 10)` |
| 전체 칸 | 2325 | 750 |
| 0인 칸 | 2250 (약 97%) | 0으로 고정된 칸 없음 |

1. 칸 수 : 단어 종류가 늘면 칸도 늘어남. 단어가 수만 개면 단어 하나가 수만 칸
2. 거리 : 모든 단어 사이 거리가 같아서 비슷한 뜻이라는 정보가 없음

- 임베딩은 칸 수를 사용자가 정하므로 단어 종류가 늘어도 길이가 그대로
- 패딩의 0은 임베딩을 써도 남음. 0번도 번호 하나라 0번 전용 벡터로 바뀔 뿐

### 1-2. Embedding 층

단어 번호마다 벡터를 한 줄씩 가진 표. 번호가 들어오면 그 줄을 꺼내 내보냄

![패딩한 문장 [0, 0, 0, 2, 3]의 번호마다 30줄 10칸짜리 표에서 같은 번호의 줄을 꺼낸다. 0 세 칸은 모두 0번 줄을 꺼내고, 꺼낸 줄을 쌓으면 (5, 10)이 된다. 파라미터는 표의 칸 수 300개다](assets/embedding-lookup.svg)

```python
Embedding(input_dim=30, output_dim=10, input_length=5)   # (15, 5) > (15, 5, 10)
```

| | 모양 | 뜻 |
|---|---|---|
| 입력 | `(15, 5)` | 문장 15개, 단어 번호 5칸 |
| 출력 | `(15, 5, 10)` | 단어 번호 하나가 `[0.03, -0.41, 0.17, ..., 0.25]` 10칸짜리 벡터로 바뀜 |

1. `input_dim` : 표의 줄 수. 단어 사전의 크기 (번호 0~29라면 30)
2. `output_dim` : 한 줄의 길이. 단어 하나를 몇 칸 벡터로 바꿀지
3. `input_length` : 문장 길이. 출력의 가운데 축이 됨

- 다른 층은 첫 인자가 출력 크기(`Dense(units)`, `LSTM(units)`)인데 Embedding은 입력(사전 크기)이 먼저
- 세 번째 자리는 `input_length`가 아니라 `embeddings_initializer`라서 `input_length`는 이름을 붙여 씀
- `input_length`를 빼면 문장 길이를 정하지 않고 받음. 출력이 `(None, None, 10)`이 되고 파라미터는 같음
- 맨 뒤에 벡터 축이 붙어서 `(15, 5)`가 `(15, 5, 10)`이 됨
- 같은 번호는 어느 문장에 있든 같은 벡터로 바뀜. 패딩 0도 0번 줄의 벡터로 바뀜
- 벡터 값은 0 근처 난수로 시작해 학습하면서 바뀌는 가중치라 0~1 같은 범위 제한이 없음

파라미터 개수는 표의 칸 수

```
input_dim × output_dim = 30 × 10 = 300
```

- `input_length`는 들어가지 않음. 문장 길이와 상관없이 같은 표에서 꺼내 씀. RNN 파라미터에 timesteps가 들어가지 않는 것과 같음 (day18 §3-4)
- bias가 없음. 곱하는 계산 없이 번호로 줄을 꺼내기만 함

## 2. Tokenizer

문장을 토큰으로 자르고 토큰마다 번호를 붙이는 케라스 도구

1. `fit_on_texts` : 문장을 읽어 단어 사전을 만듦
2. `word_index` : 단어별 번호
3. `word_counts` : 단어별 등장 횟수
4. `texts_to_sequences` : 문장을 번호 목록으로 바꿈

- 많이 나온 단어일수록 앞 번호를 받음. 횟수가 같으면 먼저 나온 순서. 번호는 1부터
- 소문자로 바꾸고 문장부호를 지움. 작은따옴표는 남아서 `it's`가 한 단어
- 여러 문장을 넣으면 사전 하나에 합쳐서 셈. 두 문장에 나온 단어는 개수가 더해져 앞 번호를 받음
- 번호는 이름표라 크기에 의미가 없는데, 그대로 넣으면 모델은 3이 2보다 크다고 계산함. 그래서 원핫 인코딩이나 임베딩을 거침

### 2-1. 원핫 전 모양

원핫은 샘플 하나당 값 하나를 벡터로 바꿈. 단어마다 원핫하려면 단어가 샘플 축(맨 앞)에 와야 함

`texts_to_sequences`는 문장 목록을 돌려주므로 문장 하나여도 `(1, 16)` 2차원

| 도구 | 받는 모양 | 넣기 전 | 결과 |
|---|---|---|---|
| `pd.get_dummies` | 1차원 | `(1, 16)` > `(16,)` | `(16, 13)` |
| `OneHotEncoder` | 2차원 `(샘플, 1)` | `(1, 16)` > `(16, 1)` | `(16, 13)` |
| `to_categorical` | 아무 차원 | 그대로 넣으면 `(1, 16, 14)` | 맨 뒤에 원핫 축이 붙음 |

- `(1, 16)`을 그대로 `OneHotEncoder`에 넣으면 에러 없이 샘플 1개, 특성 16개로 읽혀 단어마다 원핫이 되지 않음
- `to_categorical`은 0부터 칸을 만들어서 번호가 1부터면 0번 열이 빔. 패딩 0이 있는 데이터는 0도 단어처럼 원핫됨
- 문장 길이가 다르면 `np.array`로 2차원을 만들 수 없음. 문장별로 꺼내 `np.concatenate`로 이어 붙이거나 §3의 패딩으로 길이를 맞춤

## 3. 패딩

문장마다 다른 길이를 0으로 채우거나 잘라서 같은 길이로 맞추는 것

![pre는 [0, 0, 0, 2, 3]이라 실제 단어를 마지막에 읽고 바로 출력하고, post는 [2, 3, 0, 0, 0]이라 실제 단어를 읽은 뒤 0을 세 칸 더 읽고 출력한다](assets/padding-pre-post.svg)

모델은 모양이 같은 배열을 받는데 문장마다 단어 수가 달라서, 그대로는 배열을 만들 수 없음

1. `padding='pre'` : 앞쪽을 0으로 채움. 기본값
2. `padding='post'` : 뒤쪽을 0으로 채움
3. `maxlen` : 모든 문장을 맞출 길이
4. `truncating` : `maxlen`보다 긴 문장을 자를 쪽. 기본값 `'pre'`라 앞부분이 잘림

- 0을 채울 수 있는 이유 : `Tokenizer`가 번호를 1부터 매겨서 0번이 비어 있음
- 앞에 채우는 이유 : RNN은 마지막 칸 결과를 내보냄. 뒤에 채우면 실제 단어를 다 읽은 뒤 0을 더 읽고 끝나서, 그동안 순환 가중치가 더 곱해져 앞 단어의 영향이 줄어듦 (day18 §3-4)
- `maxlen`이 가장 긴 문장 이상이면 잘리는 문장이 없어서 `truncating`은 효과가 없음
- 예측할 문장도 학습 때와 같은 `maxlen`, `padding`으로 맞춤